# NUTDTS 816 Time Series Analysis
## L03 Decomposition I: models, moving averages, classical decomposition

Lab notebook for Chapter 2 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 2.1 Decomposition models

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata
ap   = tsdata.airpassengers()     # multiplicative
grid = tsdata.nigeria_grid()      # additive (simulated)
a10  = tsdata.a10()               # monthly antidiabetic drug sales, Australia: multiplicative, strong trend
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
ap.plot(ax=axes[0], title='Airline passengers: multiplicative')
grid.plot(ax=axes[1], title='Grid generation: additive')
a10.plot(ax=axes[2], title='a10 drug sales: multiplicative, growing amplitude')
for ax in axes: ax.set_xlabel('')
_caption = 'Two multiplicative series and one additive series.'

### 2.2 Estimating the trend with moving averages

In [ ]:
def centred_2xm_ma(s, m):
    """2 x m moving average (for even m) or m-MA (odd m)."""
    if m % 2 == 1:
        return s.rolling(m, center=True).mean()
    first = s.rolling(m).mean()                 # trailing m-MA
    return first.rolling(2).mean().shift(-m // 2)  # average adjacent pairs and recentre

t_hat = centred_2xm_ma(ap, 12)
ax = ap.plot(figsize=(8, 3.2), label='observed', lw=1)
t_hat.plot(ax=ax, color='#B8860B', lw=2, label='2×12-MA trend estimate')
ax.set_title('Airline passengers with a 2×12-MA trend-cycle estimate'); ax.legend(); ax.set_xlabel('')
print('First valid trend value:', t_hat.first_valid_index().date(), '| last:', t_hat.last_valid_index().date())
_caption = 'The moving average removes the annual pattern and most of the noise. Six months are lost at each end.'

### 2.3 Classical decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
dec_add  = seasonal_decompose(grid, model='additive', period=12)
dec_mult = seasonal_decompose(ap,   model='multiplicative', period=12)

fig, axes = plt.subplots(4, 2, figsize=(11, 8), sharex='col')
for col, (dec, name) in enumerate([(dec_add, 'Grid generation (additive)'), (dec_mult, 'Airline passengers (multiplicative)')]):
    for row, comp in enumerate(['observed', 'trend', 'seasonal', 'resid']):
        getattr(dec, comp).plot(ax=axes[row, col], lw=1); axes[row, col].set_ylabel(comp); axes[row, col].set_xlabel('')
    axes[0, col].set_title(name)
_caption = 'Classical decomposition panels. Note the gaps at the ends of trend and remainder, and the identical seasonal pattern every year.'

In [ ]:
print('Seasonal factors, airline (multiplicative), one year:')
print(dec_mult.seasonal['1950'].round(3).to_string())
print('\nSeasonal effects, grid (additive, MW), one year:')
print(dec_add.seasonal['2020'].round(0).to_string())

## Exercises

1. Write out the weights of a $2 \times 4$-MA and show that each quarter receives total weight $1/4$.
2. Decompose the `ausbeer` series (quarterly beer production, course data module) with classical and STL decompositions. In which years does the seasonal pattern change, and how does STL show it while classical decomposition cannot?
3. For the simulated `nigeria_grid()` series, compare the STL seasonal component with `seasonal=7` and `seasonal=25`. Which would you use for a five-year capacity plan, and why?

In [ ]:
# Your work here
